# Zenith Colab backend and dashboard

Upload **zenith-colab.zip** in step 1. The bundle includes FastAPI, the analysis worker and the updated dashboard. For a GPU demo, select **Runtime → Change runtime type → GPU** before starting.

Add `ZENITH_API_KEY` in the Colab Secrets sidebar and enable notebook access. Use a random password of at least 24 characters and keep it available to enter in the dashboard. If the secret is absent, the notebook asks privately. For an HTTPS dashboard link also add `NGROK_AUTHTOKEN`. NVIDIA is optional and uses `NVIDIA_API_KEY` only when selected.

Use a tunnel only on a runtime and plan where this use is allowed. The [Colab FAQ](https://research.google.com/colaboratory/faq.html) restricts using free managed runtimes primarily through a separate web UI. Keep the tunnel off for notebook-native processing. Runtime resets can remove uploads, evidence, downloads and the database. Use footage you have permission to process. Run cleanup when finished.

## 1. Upload and install

In [ ]:
from collections import deque
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile
import io
import os
import shutil
import signal
import subprocess
import sys
import time

from google.colab import files, userdata

if any(p.poll() is None for p in globals().get('processes', {}).values()):
    raise RuntimeError('Run the cleanup cell before reinstalling.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only zenith-colab.zip.')
root = Path('/content/zenith')
root.mkdir(exist_ok=True)
with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    names = set(archive.namelist())
    if not {'backend/main.py', 'backend/vision.py', 'dist/index.html'} <= names:
        raise ValueError('Use the matching Zenith bundle with the updated dashboard.')
    for member in archive.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root) or not member.filename.startswith(('backend/', 'dist/')):
            raise ValueError('Unexpected archive path.')
    archive.extractall(root)
del uploaded

# Preserve Colab’s installed CUDA / CPU PyTorch pair.
constraints = Path('/content/zenith-torch-constraints.txt')
constraints.write_text(f'torch=={version("torch")}\ntorchvision=={version("torchvision")}\n')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints),
    '-r', str(root / 'backend/requirements.txt'), 'pyngrok>=7.2,<8',
], check=True)
print('Zenith dependencies installed.')

## 2. Configure and launch

Start with `VISION_PROVIDER = 'off'` for YOLO + ByteTrack and the lowest setup cost. Select `nvidia` for hosted review, or `local` for a reachable OpenAI-compatible VLM. For local inference on the Colab GPU, also select `START_OLLAMA` to install Ollama and pull the 3.2 GB default model. The NVIDIA sharing setting permits hosted requests only after incident-level or run-level consent in the dashboard. Model downloads and actual reviews occur only when requested.

In [ ]:
import requests

ENABLE_PUBLIC_TUNNEL = False  # Enable only where permitted for your runtime and demo.
FRONTEND_ORIGIN = ''           # Optional separate dashboard, e.g. https://YOUR-ZENITH.vercel.app
VISION_PROVIDER = 'off'        # off | nvidia | local
ALLOW_NVIDIA_FRAME_SHARING = False
NVIDIA_VISION_MODEL = 'nvidia/nemotron-nano-12b-v2-vl'
LOCAL_VLM_BASE_URL = 'http://127.0.0.1:11434/v1'
LOCAL_VLM_MODEL = 'qwen2.5vl:3b'
START_OLLAMA = False
VISION_MAX_FRAMES = 1
VISION_TIMEOUT = 30
PRELOAD_YOLO = True

if globals().get('tunnel') or any(p.poll() is None for p in globals().get('processes', {}).values()):
    raise RuntimeError('Run cleanup before restarting Zenith.')
if VISION_PROVIDER not in {'off', 'nvidia', 'local'}:
    raise ValueError('Choose off, nvidia or local.')
if START_OLLAMA and (VISION_PROVIDER != 'local' or LOCAL_VLM_BASE_URL != 'http://127.0.0.1:11434/v1'):
    raise ValueError('For notebook-managed Ollama, select local and use the loopback URL.')
if FRONTEND_ORIGIN:
    from urllib.parse import urlsplit
    parsed = urlsplit(FRONTEND_ORIGIN)
    if (
        not parsed.hostname or parsed.username or parsed.password or parsed.path or parsed.query or parsed.fragment
        or (parsed.scheme != 'https' and (parsed.scheme != 'http' or parsed.hostname not in {'localhost', '127.0.0.1'}))
    ):
        raise ValueError('Enter the exact browser origin without a trailing slash or path.')

def colab_secret(name, prompt=None):
    try:
        value = userdata.get(name)
    except userdata.SecretNotFoundError:
        value = ''
    return value or (getpass(prompt) if prompt else '')

operator_key = colab_secret('ZENITH_API_KEY', 'Enter your random operator key (at least 24 characters): ')
if len(operator_key) < 24 or not operator_key.isascii() or any(c.isspace() for c in operator_key):
    raise ValueError('Use an operator key of at least 24 ASCII characters without spaces.')
ngrok_token = colab_secret('NGROK_AUTHTOKEN', 'Enter ngrok auth token: ') if ENABLE_PUBLIC_TUNNEL else ''
nvidia_key = colab_secret('NVIDIA_API_KEY') if VISION_PROVIDER == 'nvidia' else ''
if VISION_PROVIDER == 'nvidia' and not nvidia_key:
    raise ValueError('Add NVIDIA_API_KEY to Colab Secrets and enable notebook access.')

data = Path('/content/zenith-data')
data.mkdir(exist_ok=True)
(data / 'worker.heartbeat').unlink(missing_ok=True)
service_env = {
    **os.environ,
    'ZENITH_API_KEY': operator_key,
    'ZENITH_DATA': str(data),
    'DATABASE_URL': f'sqlite:///{data / "zenith.db"}',
    'ZENITH_ORIGINS': FRONTEND_ORIGIN,
    'ZENITH_SERVE_FRONTEND': '1',
    'ZENITH_ALLOW_LOCAL_SESSION': '0',
    'ZENITH_RECIPIENTS': '',
    'RESEND_API_KEY': '',
    'ZENITH_MODEL': 'yolov8n.pt',
    'VISION_PROVIDER': VISION_PROVIDER,
    'VISION_SEND_TO_THIRD_PARTY': str(ALLOW_NVIDIA_FRAME_SHARING).lower(),
    'NVIDIA_API_KEY': nvidia_key,
    'NVIDIA_VISION_MODEL': NVIDIA_VISION_MODEL,
    'LOCAL_VLM_BASE_URL': LOCAL_VLM_BASE_URL,
    'LOCAL_VLM_MODEL': LOCAL_VLM_MODEL,
    'LOCAL_VLM_API_KEY': colab_secret('LOCAL_VLM_API_KEY') if VISION_PROVIDER == 'local' and not START_OLLAMA else '',
    'VISION_MAX_FRAMES': str(VISION_MAX_FRAMES),
    'VISION_TIMEOUT': str(VISION_TIMEOUT),
}
provider_env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '0', 'OLLAMA_CONTEXT_LENGTH': '4096'}
processes = {}
tunnel = None
ngrok_started = False
api_origin = 'http://127.0.0.1:8000'
headers = {'Authorization': f'Bearer {operator_key}', 'ngrok-skip-browser-warning': 'zenith'}

def start_process(name, args, env):
    with Path(f'/content/zenith-{name}.log').open('w') as log:
        processes[name] = subprocess.Popen(
            args, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )

def wait_ready(url, name, predicate, headers=None):
    deadline = time.monotonic() + 45
    while time.monotonic() < deadline:
        if name and processes[name].poll() is not None:
            break
        try:
            result = requests.get(url, headers=headers, timeout=3)
            if result.ok and predicate(result):
                return
        except (requests.RequestException, ValueError):
            pass
        time.sleep(1)
    label = f' Check /content/zenith-{name}.log.' if name else ''
    raise RuntimeError(f'Service check failed for {url}.{label}')

def stop_zenith():
    global tunnel, ngrok_started
    try:
        if ngrok_started:
            from pyngrok import ngrok
            ngrok.kill()
    finally:
        tunnel = None
        ngrok_started = False
        for process in reversed(list(processes.values())):
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                except ProcessLookupError:
                    process.wait()
        processes.clear()

try:
    if PRELOAD_YOLO:
        subprocess.run([sys.executable, '-c',
                        'import torch; from ultralytics import YOLO; YOLO("yolov8n.pt"); print("CUDA:", torch.cuda.is_available())'],
                       cwd=root, check=True)
    if START_OLLAMA:
        if not shutil.which('ollama'):
            installer = Path('/content/zenith-ollama-install.sh')
            response = requests.get('https://ollama.com/install.sh', timeout=30)
            response.raise_for_status()
            installer.write_bytes(response.content)
            subprocess.run(['bash', str(installer)], check=True)
        start_process('ollama', ['ollama', 'serve'], provider_env)
        wait_ready('http://127.0.0.1:11434/api/version', 'ollama', lambda r: bool(r.json().get('version')))
        subprocess.run(['ollama', 'pull', LOCAL_VLM_MODEL], env=provider_env, check=True)

    start_process('api', [sys.executable, '-m', 'uvicorn', 'backend.main:app',
                          '--host', '127.0.0.1', '--port', '8000'], service_env)
    wait_ready(f'{api_origin}/api/health', 'api', lambda r: r.json().get('service') == 'zenith')
    start_process('worker', [sys.executable, '-m', 'backend.worker'], service_env)
    wait_ready(f'{api_origin}/api/status', 'worker', lambda r: r.json().get('worker_online'), headers)
    print('FastAPI, SQLite and analysis worker are ready.')

    if ENABLE_PUBLIC_TUNNEL:
        from pyngrok import ngrok
        ngrok.set_auth_token(ngrok_token)
        ngrok_started = True
        tunnel = ngrok.connect(addr='127.0.0.1:8000', proto='http', inspect=False)
        public_url = tunnel.public_url
        if not public_url.startswith('https://'):
            raise RuntimeError('ngrok did not return an HTTPS endpoint.')
        wait_ready(f'{public_url}/api/status', None, lambda r: r.json().get('worker_online'), headers)
        public_headers = {'ngrok-skip-browser-warning': 'zenith'}
        wait_ready(f'{public_url}/', None, lambda r: 'text/html' in r.headers.get('content-type', ''), public_headers)
        if FRONTEND_ORIGIN:
            cors = requests.options(f'{public_url}/api/status', timeout=10, headers={
                **public_headers, 'Origin': FRONTEND_ORIGIN,
                'Access-Control-Request-Method': 'GET',
                'Access-Control-Request-Headers': 'authorization,ngrok-skip-browser-warning',
            })
            cors.raise_for_status()
            if cors.headers.get('Access-Control-Allow-Origin') != FRONTEND_ORIGIN:
                raise RuntimeError('The external frontend origin did not pass the CORS check.')
        from IPython.display import Markdown, display
        display(Markdown(f'**[Open the Zenith dashboard]({public_url}/#settings)**'))
        print('On this dashboard, leave API origin blank and enter your operator key.')
        if FRONTEND_ORIGIN:
            print('For the separate frontend, enter API origin:', public_url)
    else:
        print('Tunnel is off. Use the notebook-native cells below.')
except BaseException:
    stop_zenith()
    raise

## Notebook-native video analysis
This cell uses the API locally inside the notebook, without a public web UI. Supply a real video and edit normalized polygon coordinates for your scene. Running it creates a real analysis job. Re-run the status cell manually to see progress.

In [ ]:
import requests
video_upload = files.upload()
filename, content = next(iter(video_upload.items()))
response = requests.post('http://127.0.0.1:8000/api/sources', headers=headers, files={'file': (filename, content)}, timeout=120)
response.raise_for_status()
source = response.json()
from IPython.display import Image, display
preview = requests.get(f"http://127.0.0.1:8000/api/sources/{source['id']}/preview", headers=headers, timeout=30)
preview.raise_for_status()
display(Image(preview.content, width=640))
print('Source:', source['id'], 'Edit the next cell to match this scene.')

In [ ]:
# Edit these for your scene. Never assume a zone from words alone.
POINTS = []  # e.g. [(0.2,0.2), (0.8,0.2), (0.8,0.9), (0.2,0.9)]
REVIEW_EVIDENCE = False  # Optional. Uses the provider selected in step 2.
CONSENT_TO_NVIDIA = False  # Set True only with permission to share evidence from this clip.
CONFIRM_GEOMETRY = False  # Set True only after checking your scene.
if not CONFIRM_GEOMETRY or len(POINTS) < 3:
    raise ValueError('Draw/configure and explicitly confirm your own zone first')
rule = {'capability':'zone','name':'Notebook confirmed zone','points':POINTS,'dwell_seconds':2,'confirmed':True}
job = {'source_id': source['id'], 'rule': rule}
if REVIEW_EVIDENCE and VISION_PROVIDER != 'off':
    job.update(vision_provider=VISION_PROVIDER, vision_consent=CONSENT_TO_NVIDIA)
response = requests.post('http://127.0.0.1:8000/api/runs', headers=headers, json=job, timeout=30)
response.raise_for_status()
run_id = response.json()['id']
print('Queued:', run_id)

In [ ]:
response = requests.get(f'http://127.0.0.1:8000/api/runs/{run_id}', headers=headers, timeout=30)
response.raise_for_status()
print(response.json())
events_response = requests.get(f'http://127.0.0.1:8000/api/runs/{run_id}/events', headers=headers, timeout=30)
events_response.raise_for_status()
for event in events_response.json():
    print(event['title'], 'at', event['source_seconds'], 'seconds')
    for frame in event['frames']:
        evidence = requests.get(f"http://127.0.0.1:8000/api/events/{event['id']}/evidence/{frame}", headers=headers, timeout=30)
        evidence.raise_for_status()
        display(Image(evidence.content, width=480))
    if event.get('vision_review', {}).get('status') == 'completed':
        print('Vision model interpretation:', event['vision_review']['summary'])

## Cleanup

In [ ]:
stop_zenith()
print('Zenith processes stopped and tunnel closed.')